In [ ]:
from functools import reduce
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import kmodes
import re
# plt 한글 출력 설정
plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
df_07_회원정보 = pd.read_parquet('./train/1.회원정보/201807_train_회원정보.parquet')
df_08_회원정보 = pd.read_parquet('./train/1.회원정보/201808_train_회원정보.parquet')
df_09_회원정보 = pd.read_parquet('./train/1.회원정보/201809_train_회원정보.parquet')
df_10_회원정보 = pd.read_parquet('./train/1.회원정보/201810_train_회원정보.parquet')
df_11_회원정보 = pd.read_parquet('./train/1.회원정보/201811_train_회원정보.parquet')
df_12_회원정보 = pd.read_parquet('./train/1.회원정보/201812_train_회원정보.parquet')
df_07_승인매출정보 = pd.read_parquet('./train/3.승인매출정보/201807_train_승인매출정보.parquet')
df_08_승인매출정보 = pd.read_parquet('./train/3.승인매출정보/201808_train_승인매출정보.parquet')
df_09_승인매출정보 = pd.read_parquet('./train/3.승인매출정보/201809_train_승인매출정보.parquet')
df_10_승인매출정보 = pd.read_parquet('./train/3.승인매출정보/201810_train_승인매출정보.parquet')
df_11_승인매출정보 = pd.read_parquet('./train/3.승인매출정보/201811_train_승인매출정보.parquet')
df_12_승인매출정보 = pd.read_parquet('./train/3.승인매출정보/201812_train_승인매출정보.parquet')
df_07_청구입금정보 = pd.read_parquet('./train/4.청구입금정보/201807_train_청구정보.parquet')
df_08_청구입금정보 = pd.read_parquet('./train/4.청구입금정보/201808_train_청구정보.parquet')
df_09_청구입금정보 = pd.read_parquet('./train/4.청구입금정보/201809_train_청구정보.parquet')
df_10_청구입금정보 = pd.read_parquet('./train/4.청구입금정보/201810_train_청구정보.parquet')
df_11_청구입금정보 = pd.read_parquet('./train/4.청구입금정보/201811_train_청구정보.parquet')
df_12_청구입금정보 = pd.read_parquet('./train/4.청구입금정보/201812_train_청구정보.parquet')
# train 폴더의 parquet 파일 로드
print(df_07_회원정보.shape)
print(df_07_승인매출정보.shape)
print(df_07_청구입금정보.shape)


In [ ]:
def select_columns(dfs, cols):
    return [df[cols].copy() for df in dfs]
dfs_by_category = {
    "회원정보": [df_07_회원정보, df_08_회원정보, df_09_회원정보, df_10_회원정보, df_11_회원정보, df_12_회원정보],
    "승인매출정보": [df_07_승인매출정보, df_08_승인매출정보, df_09_승인매출정보, df_10_승인매출정보, df_11_승인매출정보, df_12_승인매출정보],
    "청구입금정보": [df_07_청구입금정보, df_08_청구입금정보, df_09_청구입금정보, df_10_청구입금정보, df_11_청구입금정보, df_12_청구입금정보],
}
cols_by_category = {
    "회원정보": ['기준년월', 'ID', '남녀구분코드', '연령', '회원여부_이용가능', '소지카드수_이용가능_신용',
           '회원여부_연체', '가입통신회사코드', '거주시도명', '직장시도명', '마케팅동의여부',
           '이용카드수_신용체크', '이용카드수_신용', '이용카드수_신용_가족', '이용카드수_체크',
           '이용카드수_체크_가족', '이용금액_R3M_신용체크', '이용금액_R3M_신용', '이용금액_R3M_신용_가족',
           '이용금액_R3M_체크', '이용금액_R3M_체크_가족', '_1순위카드이용금액', '_1순위카드이용건수',
            '_1순위신용체크구분', '_2순위카드이용금액', '_2순위카드이용건수', '_2순위신용체크구분',
            '이용여부_3M_해외겸용_본인', '이용여부_3M_해외겸용_신용_본인',
            '연회비발생카드수_B0M', '기본연회비_B0M', '제휴연회비_B0M', 'Life_Stage'],
    "승인매출정보": ['기준년월', 'ID', '이용건수_신용_B0M', '이용건수_신판_B0M', '이용건수_일시불_B0M',
            '이용건수_할부_B0M', '이용건수_할부_유이자_B0M', '이용건수_할부_무이자_B0M',
            '이용건수_부분무이자_B0M', '이용건수_CA_B0M', '이용건수_체크_B0M',
            '이용건수_카드론_B0M', '이용금액_일시불_B0M', '이용금액_할부_B0M',
            '이용금액_할부_유이자_B0M', '이용금액_할부_무이자_B0M', '이용금액_부분무이자_B0M',
            '이용금액_CA_B0M', '이용금액_체크_B0M', '이용금액_카드론_B0M', '이용가맹점수',
            '이용금액_해외', '쇼핑_도소매_이용금액', '쇼핑_백화점_이용금액',
            '쇼핑_마트_이용금액', '쇼핑_슈퍼마켓_이용금액', '쇼핑_편의점_이용금액',
            '쇼핑_아울렛_이용금액', '쇼핑_온라인_이용금액', '쇼핑_기타_이용금액',
            '교통_주유이용금액', '교통_정비이용금액', '교통_통행료이용금액',
            '교통_버스지하철이용금액', '교통_택시이용금액', '교통_철도버스이용금액',
            '여유_운동이용금액', '여유_Pet이용금액', '여유_공연이용금액', '여유_공원이용금액',
            '여유_숙박이용금액', '여유_여행이용금액', '여유_항공이용금액', '여유_기타이용금액',
            '납부_통신비이용금액', '납부_관리비이용금액', '납부_렌탈료이용금액',
            '납부_가스전기료이용금액', '납부_보험료이용금액', '납부_유선방송이용금액',
            '납부_건강연금이용금액', '납부_기타이용금액', '_1순위업종', '_1순위업종_이용금액',
            '_2순위업종', '_2순위업종_이용금액', '_3순위업종', '_3순위업종_이용금액',
            '_1순위쇼핑업종', '_1순위쇼핑업종_이용금액', '_2순위쇼핑업종',
            '_2순위쇼핑업종_이용금액', '_3순위쇼핑업종', '_3순위쇼핑업종_이용금액',
            '_1순위교통업종', '_1순위교통업종_이용금액', '_2순위교통업종',
            '_2순위교통업종_이용금액', '_3순위교통업종', '_3순위교통업종_이용금액',
            '_1순위여유업종', '_1순위여유업종_이용금액', '_2순위여유업종',
            '_2순위여유업종_이용금액', '_3순위여유업종', '_3순위여유업종_이용금액',
            '_1순위납부업종', '_1순위납부업종_이용금액', '_2순위납부업종',
            '_2순위납부업종_이용금액', '_3순위납부업종', '_3순위납부업종_이용금액',
            'RP금액_B0M', 'RP건수_통신_B0M', 'RP건수_아파트_B0M',
            'RP건수_제휴사서비스직접판매_B0M', 'RP건수_렌탈_B0M', 'RP건수_가스_B0M',
            'RP건수_전기_B0M', 'RP건수_보험_B0M', 'RP건수_학습비_B0M', 'RP건수_유선방송_B0M',
            'RP건수_건강_B0M', 'RP건수_교통_B0M', '이용금액_온라인_B0M',
            '이용금액_오프라인_B0M', '이용건수_온라인_B0M', '이용건수_오프라인_B0M',
            '이용금액_페이_온라인_B0M', '이용금액_페이_오프라인_B0M',
            '이용건수_페이_온라인_B0M', '이용건수_페이_오프라인_B0M',
            '이용금액_간편결제_B0M', '이용건수_간편결제_B0M'],
    "청구입금정보": ['기준년월', 'ID', '포인트_마일리지_건별_B0M', '포인트_포인트_건별_B0M',
            '할인건수_B0M', '할인금액_B0M', '혜택수혜금액'],
}
dfs_customer = select_columns(dfs_by_category['회원정보'], cols_by_category['회원정보'])
df_07_회원정보, df_08_회원정보, df_09_회원정보, df_10_회원정보, df_11_회원정보, df_12_회원정보 = dfs_customer
dfs_sales = select_columns(dfs_by_category['승인매출정보'], cols_by_category['승인매출정보'])
df_07_승인매출정보, df_08_승인매출정보, df_09_승인매출정보, df_10_승인매출정보, df_11_승인매출정보, df_12_승인매출정보 = dfs_sales
dfs_charge = select_columns(dfs_by_category['청구입금정보'], cols_by_category['청구입금정보'])
df_07_청구입금정보, df_08_청구입금정보, df_09_청구입금정보, df_10_청구입금정보, df_11_청구입금정보, df_12_청구입금정보 = dfs_charge
print(df_07_회원정보.shape)
print(df_07_승인매출정보.shape)
print(df_07_청구입금정보.shape)


In [ ]:
df_07 = reduce(lambda x, y : pd.merge(x, y, on=['기준년월', 'ID'], how='left'), [df_07_회원정보, df_07_승인매출정보, df_07_청구입금정보])
df_08 = reduce(lambda x, y : pd.merge(x, y, on=['기준년월', 'ID'], how='left'), [df_08_회원정보, df_08_승인매출정보, df_08_청구입금정보])
df_09 = reduce(lambda x, y : pd.merge(x, y, on=['기준년월', 'ID'], how='left'), [df_09_회원정보, df_09_승인매출정보, df_09_청구입금정보])
df_10 = reduce(lambda x, y : pd.merge(x, y, on=['기준년월', 'ID'], how='left'), [df_10_회원정보, df_10_승인매출정보, df_10_청구입금정보])
df_11 = reduce(lambda x, y : pd.merge(x, y, on=['기준년월', 'ID'], how='left'), [df_11_회원정보, df_11_승인매출정보, df_11_청구입금정보])
df_12 = reduce(lambda x, y : pd.merge(x, y, on=['기준년월', 'ID'], how='left'), [df_12_회원정보, df_12_승인매출정보, df_12_청구입금정보])
df_all = pd.concat([df_07, df_08, df_09, df_10, df_11, df_12], axis = 0)
print(df_07.shape)
print(df_all.shape)
print(df_all.head())
df_07.to_csv('./merged/merged_07.csv', encoding = 'utf-8-sig', index=False)
df_08.to_csv('./merged/merged_08.csv', encoding = 'utf-8-sig', index=False)
df_09.to_csv('./merged/merged_09.csv', encoding = 'utf-8-sig', index=False)
df_10.to_csv('./merged/merged_10.csv', encoding = 'utf-8-sig', index=False)
df_11.to_csv('./merged/merged_11.csv', encoding = 'utf-8-sig', index=False)
df_12.to_csv('./merged/merged_12.csv', encoding = 'utf-8-sig', index=False)
df_all.to_csv('./merged/merged_all.csv', encoding = 'utf-8-sig', index=False)


In [ ]:
df_07 = pd.read_csv('./merged/merged_07.csv', encoding = 'utf-8-sig')
df_08 = pd.read_csv('./merged/merged_08.csv', encoding = 'utf-8-sig')
df_09 = pd.read_csv('./merged/merged_09.csv', encoding = 'utf-8-sig')
df_10 = pd.read_csv('./merged/merged_10.csv', encoding = 'utf-8-sig')
df_11 = pd.read_csv('./merged/merged_11.csv', encoding = 'utf-8-sig')
df_12 = pd.read_csv('./merged/merged_12.csv', encoding = 'utf-8-sig')
df_all = pd.read_csv('./merged/merged_all.csv', encoding = 'utf-8-sig')
print(df_all.head())
print(df_all.shape)
print(df_all['기준년월'].value_counts())


In [ ]:
print(df_07['남녀구분코드'].value_counts())
print(df_07['연령'].value_counts())
# 표본의 개수는 총 40만개, 남녀 비율은 거의 동일, 연령의 경우 40대가 가장 많았으며 40대에서 멀어질수록 비율이 감소했음
print(df_07['거주시도명'].value_counts())
print(df_07['직장시도명'].value_counts())
# 거주시도명 및 직장시도명의 경우 절반 이상이 수도권
# 전국 인구 분포와 비교시 서울/대전/충북 과대표집, 비수도권 지역 대체로 과소표집
print(df_07['회원여부_이용가능'].value_counts())
print(df_07['회원여부_연체'].value_counts())
# 블랙리스트 등재 고객은 표본의 최대 약 5%, 연체 고객은 표본의 최대 약 2% 수준
# 블랙리스트 등재 또는 연체 고객에게는 추천 신용카드를 출력하지 않는등 신용도를 고려한 추천 시스템 구축 필요
print(df_07['이용카드수_신용'].describe())
print(df_07['이용카드수_체크'].describe())
print(df_07['이용금액_R3M_신용'].describe())
print(df_07['이용금액_R3M_체크'].describe())
# 대체로 이용하는 신용카드의 수가 체크카드의 수보다 많았음
# 대다수(Q1 ~ Q3) 고객은 신용카드 1~2개, 체크카드 0개를 이용했음
sns.histplot(df_07['이용금액_R3M_신용'], bins=100, color='orange')
plt.xlabel('Credit')
plt.ylim(0 ,40000)
plt.show()
sns.histplot(df_07['이용금액_R3M_체크'], bins=100, color='blue')
plt.xlabel('Check')
plt.ylim(0, 25000)
plt.show()
# 신용카드의 경우 최근 3개월간 이용금액이 0원인 경우가 최빈값이었으며 이를 제외하더라도 오른꼬리 분포를 이룸
# 사용자군에 따른 심리적/사회학적 특성 고려하여 추천 카드의 종류 달리해야
# 체크카드 사용금액이 매우 낮으나 신용카드 이용액인 0인 고객의 존재를 생각해보았을때 신규 신용카드 추천 보조 지표로 활용
sns.histplot(df_07['_1순위카드이용금액'], bins=100, color='orange')
plt.xlabel('First')
plt.ylim(0 ,30000)
plt.show()
print(df_07['_1순위신용체크구분'].value_counts())
sns.histplot(df_07['_2순위카드이용금액'], bins=100, color='blue')
plt.xlabel('Second')
plt.ylim(0 ,10000)
plt.show()
print(df_07['_2순위신용체크구분'].value_counts())
# 1순위 카드는 신용카드가 압도적이나 2순위 카드는 체크카드의 비중이 다소 있었음
# 주 카드로 신용카드를 사용하고 체크카드를 보조적으로 사용하는 경향성 존재
# 체크카드의 보조적 가치
sns.histplot(df_07['연회비_B0M'], bins=100, color='orange')
plt.xlabel('Annual Fee')
plt.show()
print(df_07['연회비_B0M'].describe())
# 연회비 무료 카드가 대다수, 기존 카드의 연회비 고려하여 신용카드 추천하여야 함
print(df_07['Life_Stage'].value_counts())
print(df_07['Segment'].value_counts())
# 생애주기의 경우 자녀성장기 - 자녀출산기 - 가족구축기 순서로 비율이 높았음
# 남녀별 소비패턴 차이
plt.figure(figsize=(12, 6))
sns.histplot(data=df_07, x='이용건수_신판_B0M', hue='남녀구분코드', multiple='stack', bins=100)
plt.xlabel('Credit B0M')
plt.show()
plt.figure(figsize=(12, 6))
sns.histplot(data=df_07, x='이용건수_체크_B0M', hue='남녀구분코드', multiple='stack', bins=100)
plt.xlabel('Check B0M')
plt.ylim(0, 5000)
plt.show()
plt.figure(figsize=(12, 6))
sns.histplot(data=df_07, x='거래당평균이용금액_신판_B0M', hue='남녀구분코드', multiple='stack', bins=100)
plt.xlabel('Credit B0M')
plt.show()
sns.histplot(data=df_07, x='거래당평균이용금액_체크_B0M', hue='남녀구분코드', multiple='stack', bins=100)
plt.xlabel('Check B0M')
plt.ylim(0, 5000)
plt.show()
print(df_07.groupby('남녀구분코드')['거래당평균이용금액_신판_B0M'].describe())
print(df_07.groupby('남녀구분코드')['거래당평균이용금액_체크_B0M'].describe())
# 1인 경우 이용건수가 더 많고 거래당 이용액은 차이가 없었음
def calculate_top10_average_usage(df_07):
    long_frames = []
    pairs = [('_1순위업종', '_1순위업종_이용금액'), ('_2순위업종', '_2순위업종_이용금액'), ('_3순위업종', '_3순위업종_이용금액'), ('_1순위쇼핑업종', '_1순위쇼핑업종_이용금액'), ('_2순위쇼핑업종', '_2순위쇼핑업종_이용금액'), ('_3순위쇼핑업종', '_3순위쇼핑업종_이용금액'), ('_1순위교통업종', '_1순위교통업종_이용금액'), ('_2순위교통업종', '_2순위교통업종_이용금액'), ('_3순위교통업종', '_3순위교통업종_이용금액'), ('_1순위여유업종', '_1순위여유업종_이용금액'), ('_2순위여유업종', '_2순위여유업종_이용금액'), ('_3순위여유업종', '_3순위여유업종_이용금액'), ('_1순위납부업종', '_1순위납부업종_이용금액'), ('_2순위납부업종', '_2순위납부업종_이용금액'), ('_3순위납부업종', '_3순위납부업종_이용금액')]

    for upjong_col, amt_col in pairs:
        tmp = df_07[['남녀구분코드', upjong_col, amt_col]].copy()
        tmp.columns = ['성별', '업종', '이용금액']
        long_frames.append(tmp)

    long_df = pd.concat(long_frames, ignore_index=True)
    long_df = long_df.dropna(subset=['이용금액'])
    long_df = long_df[long_df['이용금액'] > 0]

    avg_amt = long_df.groupby(['성별', '업종'], as_index=False)['이용금액'].mean()
    top10_each_gender = avg_amt.sort_values(['성별', '이용금액'], ascending=[True, False]).groupby('성별').head(10).copy()
    top10_each_gender['순위'] = top10_each_gender.groupby('성별')['이용금액'].rank(method='first', ascending=False).astype(int)

    male = top10_each_gender[top10_each_gender['성별'] == 1].sort_values('순위')[['순위', '업종', '이용금액']].reset_index(drop=True)
    female = top10_each_gender[top10_each_gender['성별'] == 2].sort_values('순위')[['순위', '업종', '이용금액']].reset_index(drop=True)
    combined = pd.concat([male, female], axis=1, keys=[1, 2])

    return combined
result_top10_average_usage = calculate_top10_average_usage(df_07)
print(result_top10_average_usage)
# 남녀코드별 상위 이용 업종 조회, 경향성은 유사하나 일부 차이점 존재
from scipy.stats import skew
def calculate_zero_ratio_and_skewness(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) == 0:
        print("수치형 변수가 없습니다.")
        return pd.DataFrame()

    results = []
    for col in numeric_cols:
        zero_ratio = (df[col] == 0).mean()
        skewness = skew(df[col].dropna())  # 결측값 제거 후 왜도 계산

        results.append({
            'column': col,
            'zero_ratio': zero_ratio,
            'skewness': skewness
        })

    result_df = pd.DataFrame(results).set_index('column')
    result_df['zero_ratio'] = result_df['zero_ratio'].round(4)
    result_df['skewness'] = result_df['skewness'].round(4)

    return result_df

result_zero_ratio_and_skewness = calculate_zero_ratio_and_skewness(df_all)
print(analysis_result_zero_ratio_and_skewness)
# 대다수 컬럼의 경우 0비율이 상당히 높았으며 왜도 또한 높았음


In [ ]:
df_all_customer = pd.read_csv('./merged/merged_all_scaled.csv',encoding = 'utf-8-sig')
df_all_card = pd.read_excel('./merged/merged_card.xlsx')
# 데이터 로드
from sklearn.preprocessing import OrdinalEncoder
life_stage_order = ['독신', '가족구축기', '자녀출산기', '자녀성장(1)', '자녀성장(2)', '자녀독립기', '노년생활']
life_stage_encoder = OrdinalEncoder(categories=[life_stage_order])
print(df_all_customer['Life_Stage'].value_counts())
df_all_customer['Life_Stage'] = life_stage_encoder.fit_transform(df_all_customer[['Life_Stage']]).astype(int)
print(df_all_customer['Life_Stage'].value_counts())
# 고객 데이터 Life_Stage 컬럼 수치형 변환
df_all_customer['연령'] = df_all_customer['연령'].str.extract(r'(\d+)').astype(int)
# 고객 데이터 연령 컬럼 수치형 변환


In [ ]:
df_clustering_customer = df_all_customer[['남녀구분코드', '연령', 'Life_Stage', '이용금액_해외',
                                            '쇼핑_도소매_이용금액', '쇼핑_백화점_이용금액', '쇼핑_마트_이용금액', '쇼핑_슈퍼마켓_이용금액',
                                            '쇼핑_편의점_이용금액',  '쇼핑_아울렛_이용금액', '쇼핑_온라인_이용금액', '쇼핑_기타_이용금액', '교통_주유이용금액',
                                            '교통_정비이용금액', '교통_통행료이용금액', '교통_버스지하철이용금액', '교통_택시이용금액', '교통_철도버스이용금액',
                                            '여유_운동이용금액', '여유_Pet이용금액', '여유_공연이용금액', '여유_공원이용금액', '여유_숙박이용금액',
                                            '여유_여행이용금액',  '여유_항공이용금액', '여유_기타이용금액', '납부_통신비이용금액', '납부_관리비이용금액',
                                            '납부_렌탈료이용금액', '납부_가스전기료이용금액', '납부_보험료이용금액', '납부_유선방송이용금액', '납부_건강연금이용금액',
                                            '납부_기타이용금액', '이용금액_온라인_B0M', '이용금액_오프라인_B0M', '이용금액_페이_온라인_B0M',
                                            '이용금액_페이_오프라인_B0M', '할인금액_B0M', '혜택수혜금액', '이용금액_신판_B0M', '총연회비_B0M']].copy()
df_clustering_customer['할인혜택선호율'] = np.where(df_clustering_customer['혜택수혜금액'] != 0,
                                        df_clustering_customer['할인금액_B0M'] / df_clustering_customer['혜택수혜금액'], 0)
df_clustering_customer['할인혜택선호율'] = np.where(df_clustering_customer['할인혜택선호율'] > 1, 1, df_clustering_customer['할인혜택선호율'])
df_clustering_customer['대중교통소비액'] = df_clustering_customer[['교통_버스지하철이용금액', '교통_택시이용금액', '교통_철도버스이용금액']].sum(axis = 1)
df_clustering_customer['자가용소비액'] = df_clustering_customer[['교통_주유이용금액', '교통_정비이용금액', '교통_통행료이용금액']].sum(axis = 1)
df_clustering_customer['여행소비액'] = df_clustering_customer['여유_여행이용금액'] + df_clustering_customer['여유_숙박이용금액'] + df_clustering_customer['여유_항공이용금액'] / 2
df_clustering_customer['문화생활소비액'] = df_clustering_customer[['여유_공연이용금액', '여유_운동이용금액', '여유_공원이용금액', '여유_기타이용금액']].sum(axis = 1)
df_clustering_customer['쇼핑소비액'] = (df_clustering_customer['쇼핑_도소매_이용금액'] / 2 + df_clustering_customer['쇼핑_백화점_이용금액'] +
                                   df_clustering_customer['쇼핑_아울렛_이용금액'] + df_clustering_customer['쇼핑_온라인_이용금액'] / 2)
df_clustering_customer['생필품소비액'] = (df_clustering_customer['쇼핑_도소매_이용금액'] / 2 + df_clustering_customer['쇼핑_마트_이용금액'] +
                                   df_clustering_customer['쇼핑_편의점_이용금액'] + df_clustering_customer['쇼핑_슈퍼마켓_이용금액'] +
                                    df_clustering_customer['쇼핑_온라인_이용금액'] / 2 + df_clustering_customer['쇼핑_기타_이용금액'])
df_clustering_customer['정기납부액'] = (df_clustering_customer[['납부_렌탈료이용금액', '납부_통신비이용금액', '납부_관리비이용금액', '납부_가스전기료이용금액',
                                   '납부_유선방송이용금액', '납부_기타이용금액']].sum(axis = 1) + df_clustering_customer['납부_보험료이용금액']
                                   / 2 + df_clustering_customer['납부_건강연금이용금액'] / 2 )
df_clustering_customer['디지털소비액'] = df_clustering_customer[['이용금액_온라인_B0M', '이용금액_페이_온라인_B0M', '이용금액_페이_오프라인_B0M']].sum(axis = 1)
df_clustering_customer['가족관련소비액'] = (df_clustering_customer['납부_건강연금이용금액'] / 2 + df_clustering_customer['납부_보험료이용금액']
                                     / 2 + df_clustering_customer['여유_Pet이용금액'])
df_clustering_customer = df_clustering_customer[df_clustering_customer['할인혜택선호율'] <= 1]
df_clustering_customer = df_clustering_customer[['남녀구분코드', '연령', 'Life_Stage', '할인혜택선호율', '대중교통소비액', '자가용소비액',
                                                 '여행소비액', '문화생활소비액', '쇼핑소비액', '생필품소비액', '정기납부액', '디지털소비액',
                                                 '가족관련소비액', '이용금액_해외', '이용금액_신판_B0M', '총연회비_B0M']]
# 군집화 용 데이터프레임 생성 및 20개 이내로 차원축소
print(df_clustering_customer.columns)


In [ ]:
df_clustering_customer_reduced = df_clustering_customer.copy()
df_clustering_customer_reduced['교통소비액'] = df_clustering_customer_reduced['대중교통소비액'] + df_clustering_customer_reduced['자가용소비액']
df_clustering_customer_reduced['사치소비액'] = df_clustering_customer_reduced['여행소비액'] + df_clustering_customer_reduced[
    '문화생활소비액'] + df_clustering_customer_reduced['쇼핑소비액']
df_clustering_customer_reduced['생활소비액'] = df_clustering_customer_reduced['생필품소비액'] + df_clustering_customer_reduced['정기납부액']
df_clustering_customer_reduced = df_clustering_customer_reduced[['남녀구분코드', '연령', 'Life_Stage', '할인혜택선호율', '교통소비액', '사치소비액',
                                                                 '생활소비액', '디지털소비액', '가족관련소비액', '이용금액_해외', '이용금액_신판_B0M', '총연회비_B0M']]
print(df_clustering_customer_reduced.columns)


In [ ]:
from sklearn.model_selection import train_test_split
df_clustering_customer['stratifyKey'] = (df_clustering_customer['남녀구분코드'].astype(str) + '_' + df_clustering_customer['연령'].astype(str) + '_')
def dataframe_sampling(df, sample_size, stratify_col):
    train_idx, sample_idx = train_test_split(np.arange(len(df)),
        test_size = sample_size,
        stratify = df[stratify_col],
        random_state = 42
    )
    df_sample = df.iloc[sample_idx]
    return df_sample
# 대표 샘플 추출 함수
df_clustering_customer_sampled = dataframe_sampling(df_clustering_customer, 18755, 'stratifyKey')
print(df_clustering_customer['남녀구분코드'].value_counts())
print(df_clustering_customer['연령'].value_counts())
print(df_clustering_customer['Life_Stage'].value_counts())
print(df_clustering_customer_sampled['남녀구분코드'].value_counts())
print(df_clustering_customer_sampled['연령'].value_counts())
print(df_clustering_customer_sampled['Life_Stage'].value_counts())
print(df_clustering_customer['이용금액_신판_B0M'].describe())
print(df_clustering_customer['총연회비_B0M'].describe())
print(df_clustering_customer['이용금액_해외'].describe())
print(df_clustering_customer_sampled['이용금액_신판_B0M'].describe())
print(df_clustering_customer_sampled['총연회비_B0M'].describe())
print(df_clustering_customer_sampled['이용금액_해외'].describe())
df_clustering_customer = df_clustering_customer.drop('stratifyKey', axis = 1)
df_clustering_customer_sampled = df_clustering_customer_sampled.drop('stratifyKey', axis = 1)


In [ ]:
df_clustering_customer_reduced['stratifyKey'] = (df_clustering_customer_reduced['남녀구분코드'].astype(str) + '_' +
                                                 df_clustering_customer_reduced['연령'].astype(str) + '_')
df_clustering_customer_reduced_sampled = dataframe_sampling(df_clustering_customer_reduced, 18755, 'stratifyKey')
df_clustering_customer_reduced = df_clustering_customer_reduced.drop('stratifyKey', axis = 1)
df_clustering_customer_reduced_sampled = df_clustering_customer_reduced_sampled.drop('stratifyKey', axis = 1)


In [ ]:
df_all_customer['stratifyKey'] = (df_all_customer['남녀구분코드'].astype(str) + '_' +
                                                 df_all_customer['연령'].astype(str) + '_')
df_all_customer_sampled = dataframe_sampling(df_all_customer, 18755, 'stratifyKey')
df_all_customer = df_all_customer.drop('stratifyKey', axis = 1)
df_all_customer_sampled = df_all_customer_sampled.drop(['stratifyKey', '기준년월', 'ID', '대중교통_점수', '자가용_점수', '해외_점수', '여행_점수',
       '문화생활_점수', '쇼핑_점수', '생필품_점수', '납부(고정지출)_점수', '디지털결제_점수', '가족_점수', '이용여부_3M_해외겸용_본인'], axis = 1)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
def visualize_optimal_k(df, k_range=range(2, 11)):
    # 수치형 컬럼만 선택
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if len(numeric_cols) == 0:
        print("오류: 수치형 컬럼이 없습니다. K-means는 수치형 데이터만 사용 가능합니다.")
        return None, None

    print(f"수치형 컬럼 ({len(numeric_cols)}개): {numeric_cols}")

    # 수치형 데이터만 추출
    X = df[numeric_cols].values

    # 결측치 처리
    if pd.DataFrame(X).isnull().sum().sum() > 0:
        print("경고: 결측치가 발견되어 0으로 채웁니다.")
        X = pd.DataFrame(X).fillna(0).values

    print(f"데이터 형태: {X.shape}")

    # 결과 저장용 리스트
    inertias = []  # Within-cluster sum of squares (WCSS)
    silhouette_scores = []

    print("\nK-means 클러스터링 성능 평가 중...")

    for k in k_range:
        print(f"K={k} 처리 중...")

        if df.shape[0] > 10000 :
            # K-means 클러스터링
            kmeans = KMeans(
                n_clusters=k,
                init='k-means++',
                random_state=42,
                n_init=10,
                max_iter=300
            )
        else :
            kmeans = MiniBatchKMeans(
                n_clusters=k,
                init='k-means++',
                batch_size=1000,
                random_state=42,
                n_init=10,
                max_iter=300
            )

        labels = kmeans.fit_predict(X)

        # 1) 엘보우 기법: Inertia (WCSS - Within-Cluster Sum of Squares)
        inertias.append(kmeans.inertia_)

        # 2) 실루엣 점수
        try:
            silhouette_avg = silhouette_score(X, labels, metric='euclidean')
            silhouette_scores.append(silhouette_avg)
            print(f"K={k}: Inertia = {kmeans.inertia_:.2f}, Silhouette Score = {silhouette_avg:.3f}")
        except Exception as e:
            print(f"실루엣 점수 계산 실패 (K={k}): {e}")
            silhouette_scores.append(np.nan)

    # 시각화
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # 왼쪽 축: Inertia (엘보우 기법)
    ax1.set_xlabel('Number of Clusters (k)')
    ax1.set_ylabel('Inertia (WCSS - lower is better)', color='tab:blue')
    line1 = ax1.plot(k_range, inertias, 'o-', color='tab:blue', linewidth=2, markersize=6, label='Inertia (WCSS)')
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    ax1.grid(True, alpha=0.3)

    # 오른쪽 축: 실루엣 점수
    ax2 = ax1.twinx()
    ax2.set_ylabel('Silhouette Score (higher is better)', color='tab:green')
    line2 = ax2.plot(k_range, silhouette_scores, 's--', color='tab:green', linewidth=2, markersize=6,
                     label='Silhouette Score')
    ax2.tick_params(axis='y', labelcolor='tab:green')

    # 범례
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    plt.title('Elbow Method (Inertia) vs Silhouette Score for Optimal K\n(K-means Clustering)',
              fontsize=14)
    plt.tight_layout()
    plt.show()

    return inertias, silhouette_scores
# 모든 컬럼에 대하여 엘보우 기법으로, 수치형 컬럼에 대하여 실루엣 계수로 시각화

In [ ]:
try:
    inertias_sampled, silhouette_sampled =  visualize_optimal_k(df_clustering_customer_sampled)
    print("실행 완료!")
except Exception as e:
    print(f"에러 발생: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
inertias_sampled_all, silhouette_sampled_all = visualize_optimal_k(df_all_customer_sampled)


In [ ]:
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

def visualize_optimal_agglomerative(df, range_n_clusters=range(2, 11), linkage_method='ward'):
    # 1. 데이터 전처리
    X = df.select_dtypes(include=[np.number]).values
    selected_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"자동 선택된 컬럼: {selected_columns}")

    # 결측치 처리
    if pd.DataFrame(X).isnull().sum().sum() > 0:
        print("경고: 결측치가 발견되어 0으로 채웁니다.")
        X = pd.DataFrame(X).fillna(0).values

    print(f"데이터 형태: {X.shape}")

    # 준비
    silhouette_scores = []
    calinski_harabasz_scores = []
    davies_bouldin_scores = []

    print("\n클러스터링 성능 평가 중...")

    # 2. 각 군집 수에 대해 평가지표 계산
    for n_clusters in range_n_clusters:
        clustering = AgglomerativeClustering(
            n_clusters=n_clusters,
            linkage=linkage_method
        )
        cluster_labels = clustering.fit_predict(X)

        silhouette_avg = silhouette_score(X, cluster_labels)
        silhouette_scores.append(silhouette_avg)

        calinski_harabasz_avg = calinski_harabasz_score(X, cluster_labels)
        calinski_harabasz_scores.append(calinski_harabasz_avg)

        davies_bouldin_avg = davies_bouldin_score(X, cluster_labels)
        davies_bouldin_scores.append(davies_bouldin_avg)

        print(f"군집수 {n_clusters}: 실루엣 점수 = {silhouette_avg:.3f}, CH 지수 = {calinski_harabasz_avg:.3f}, DB 지수 = {davies_bouldin_avg:.3f}")

    # 3. 세 평가지표 그래프 시각화
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 6))

    # 실루엣 점수
    ax1.plot(range_n_clusters, silhouette_scores, 'o-', color='blue', linewidth=2, markersize=8)
    ax1.set_xlabel('군집수')
    ax1.set_ylabel('실루엣 점수')
    ax1.set_title('실루엣 점수에 따른 최적 군집수')
    ax1.grid(True, alpha=0.3)

    # 칼린스키-하라바즈 지수
    ax2.plot(range_n_clusters, calinski_harabasz_scores, 's-', color='red', linewidth=2, markersize=8)
    ax2.set_xlabel('군집수')
    ax2.set_ylabel('칼린스키-하라바즈 지수')
    ax2.set_title('칼린스키-하라바즈 지수에 따른 최적 군집수')
    ax2.grid(True, alpha=0.3)

    # 데이비스-볼딘 지수
    ax3.plot(range_n_clusters, davies_bouldin_scores, '^-', color='green', linewidth=2, markersize=8)
    ax3.set_xlabel('군집수')
    ax3.set_ylabel('데이비스-볼딘 지수')
    ax3.set_title('데이비스-볼딘 지수에 따른 최적 군집수')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # 4. 최적 군집수 결정
    best_silhouette_n = list(range_n_clusters)[np.argmax(silhouette_scores)]
    best_ch_n = list(range_n_clusters)[np.argmax(calinski_harabasz_scores)]
    best_db_n = list(range_n_clusters)[np.argmin(davies_bouldin_scores)]

    # 5. 덴드로그램 시각화를 위한 헬퍼 함수
    def plot_dendrogram_from_model(X, n_clusters, title):
        model = AgglomerativeClustering(
            n_clusters=None,
            distance_threshold=0,
            linkage=linkage_method
        )
        model = model.fit(X)

        counts = np.zeros(model.children_.shape[0])
        n_samples = len(model.labels_)

        for i, merge in enumerate(model.children_):
            current_count = 0
            for child_idx in merge:
                if child_idx < n_samples:
                    current_count += 1
                else:
                    current_count += counts[child_idx - n_samples]
            counts[i] = current_count

        linkage_matrix = np.column_stack([
            model.children_,
            model.distances_,
            counts
        ]).astype(float)

        plt.figure(figsize=(12, 8))
        dendrogram(
            linkage_matrix,
            truncate_mode='level',
            p=5,
            leaf_rotation=90,
            leaf_font_size=10
        )
        plt.title(f'{title}', fontsize=14)
        plt.xlabel('샘플 인덱스 또는 클러스터 크기')
        plt.ylabel('거리')
        plt.tight_layout()
        plt.show()

    # 6. 최적 군집수에 따른 덴드로그램 시각화
    if best_silhouette_n == best_ch_n == best_db_n:
        print(f"\n세 지표가 모두 {best_silhouette_n}개 군집을 최적으로 선택했습니다.")
        plot_dendrogram_from_model(X, best_silhouette_n, "세 지표 기준 최적 군집수 덴드로그램")
    else:
        print(f"\n최적 군집수가 각기 다릅니다. 각 지표별로 덴드로그램을 표시합니다.")
        plot_dendrogram_from_model(X, best_silhouette_n, "실루엣 점수 기준 최적 군집수 덴드로그램")
        plot_dendrogram_from_model(X, best_ch_n, "칼린스키-하라바즈 지수 기준 최적 군집수 덴드로그램")
        plot_dendrogram_from_model(X, best_db_n, "데이비스-볼딘 지수 기준 최적 군집수 덴드로그램")

    return silhouette_scores, calinski_harabasz_scores, davies_bouldin_scores

In [ ]:
import time
try:
    start_time = time.time()
    silhouette_sample, ch_sample, db_sample = visualize_optimal_agglomerative(df_clustering_customer_sampled)
    finish_time = time.time()
    print(f"실행 시간: {finish_time - start_time:.6f}초")
except Exception as e:
    print(f"에러 발생: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
silhouette_all, ch_all, db_all = visualize_optimal_agglomerative(df_all_customer_sampled)